In [25]:

import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
import torch


In [26]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from itertools import cycle
import gc
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score, precision_score, recall_score, f1_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
import pandas as pd
import seaborn as sns
from itertools import cycle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
DATASET_PATH = "/content/drive/MyDrive/brain_tumor/data"   # train / val / test
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-4

In [29]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),

    transforms.RandomAffine(
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR
    ),

    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.2)
    ], p=0.4),

    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.1),

    transforms.ToTensor(),

    transforms.RandomApply([
        transforms.Lambda(lambda x: x + 0.02 * torch.randn_like(x))
    ], p=0.3),

    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [30]:
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [31]:
from torch.utils.data import random_split, Subset

train_dir = os.path.join(DATASET_PATH, "Train")
test_dir = os.path.join(DATASET_PATH, "Test")

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=val_transform
)

full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
full_val_dataset_source = datasets.ImageFolder(train_dir, transform=val_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

train_dataset = train_subset

val_dataset = Subset(full_val_dataset_source, val_subset.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(full_train_dataset.classes)
class_names = full_train_dataset.classes

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Classes: {class_names}")

Train dataset size: 7720
Val dataset size: 1930
Test dataset size: 2414
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


***EfficientNet-B0 + MobileNetV2***

In [32]:
class EfficientNet_MobileNet_Hybrid(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super().__init__()

        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.mobilenet = models.mobilenet_v2(pretrained=True)

        self.efficientnet_features = self.efficientnet.features
        self.mobilenet_features = self.mobilenet.features

        self.pool = nn.AdaptiveAvgPool2d((1, 1))


        fused_dim = 1280 + 1280

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        for param in self.efficientnet_features.parameters():
            param.requires_grad = False
        for param in self.mobilenet_features.parameters():
            param.requires_grad = False

        for param in self.efficientnet_features[-2:].parameters():
            param.requires_grad = True
        for param in self.mobilenet_features[-2:].parameters():
            param.requires_grad = True

    def forward(self, x):
        ef = self.efficientnet_features(x)
        ef = self.pool(ef)
        ef = ef.view(ef.size(0), -1)

        mb = self.mobilenet_features(x)
        mb = self.pool(mb)
        mb = mb.view(mb.size(0), -1)


        fused = torch.cat((ef, mb), dim=1)

        return self.classifier(fused)


**Multi-Scale EfficientNet (Early + Late Fusion)**

In [33]:
class MultiScale_EffNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        eff = models.efficientnet_b0(pretrained=True).features

        self.stem = eff[0]

        self.low  = nn.Sequential(*eff[1:3])
        self.mid  = nn.Sequential(*eff[3:5])
        self.high = nn.Sequential(*eff[5:])

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(24 + 80 + 1280, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)

        l_feat = self.low(x)
        m_feat = self.mid(l_feat)
        h_feat = self.high(m_feat)

        l = self.pool(l_feat).flatten(1)
        m = self.pool(m_feat).flatten(1)
        h = self.pool(h_feat).flatten(1)

        return self.fc(torch.cat((l, m, h), dim=1))

**EfficientNet-B0 + CBAM + LSTM**




In [34]:
class EffNet_CBAM_LSTM(nn.Module):
    def __init__(self, num_classes, hidden=256):
        super().__init__()

        self.eff = models.efficientnet_b0(pretrained=True).features
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.attn = nn.MultiheadAttention(1280, 8, batch_first=True)
        self.lstm = nn.LSTM(1280, hidden, batch_first=True)

        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):
        f = self.pool(self.eff(x)).flatten(1).unsqueeze(1)
        f, _ = self.attn(f, f, f)
        o, _ = self.lstm(f)
        return self.fc(o[:, -1, :])


**DenseNet121 + SE-Attention + LSTM**

In [35]:
class DenseNet_SE_LSTM(nn.Module):
    def __init__(self, num_classes, hidden=256):
        super().__init__()

        self.den = models.densenet121(pretrained=True).features
        self.pool = nn.AdaptiveAvgPool2d(1)

        # Squeeze & Excitation
        self.se = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 1024),
            nn.Sigmoid()
        )

        self.lstm = nn.LSTM(1024, hidden, batch_first=True)

        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x):
        f = self.pool(self.den(x)).flatten(1)

        se_weight = self.se(f)
        f = f * se_weight

        f = f.unsqueeze(1)
        o, _ = self.lstm(f)

        return self.fc(o[:, -1, :])


**EfficientNet-B0 + Spatial Attention + Residual Fusion**

In [36]:
class EffNet_SpatialAttention_Residual(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.eff = models.efficientnet_b0(pretrained=True).features

        self.spatial_attn = nn.Sequential(
            nn.Conv2d(1280, 256, 1),
            nn.ReLU(),
            nn.Conv2d(256, 1, 1),
            nn.Sigmoid()
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.BatchNorm1d(1280),
            nn.Dropout(0.5),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        f = self.eff(x)
        attn = self.spatial_attn(f)
        f = f * attn + f  # residual

        f = self.pool(f).flatten(1)
        return self.fc(f)


**Hierarchical CNN + Attention Pooling**

In [37]:
class Hierarchical_CNN_AttnPool(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        res = models.resnet18(pretrained=True)

        self.layer1 = nn.Sequential(*list(res.children())[:5])
        self.layer2 = res.layer2
        self.layer3 = res.layer3
        self.layer4 = res.layer4

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.attn_fc = nn.Sequential(
            nn.Linear(64 + 128 + 256 + 512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Linear(64 + 128 + 256 + 512, num_classes)

    def forward(self, x):
        f1 = self.pool(self.layer1(x)).flatten(1)
        f2 = self.pool(self.layer2(self.layer1(x))).flatten(1)
        f3 = self.pool(self.layer3(self.layer2(self.layer1(x)))).flatten(1)
        f4 = self.pool(self.layer4(self.layer3(self.layer2(self.layer1(x))))).flatten(1)

        feats = torch.cat([f1, f2, f3, f4], dim=1)

        return self.classifier(feats)


**EfficientNet-B0 + ResNet50**

In [38]:
class EffNet_ResNet_CrossAttention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Backbones
        self.eff = models.efficientnet_b0(pretrained=True).features
        self.res = nn.Sequential(*list(models.resnet50(pretrained=True).children())[:-2])

        self.pool = nn.AdaptiveAvgPool2d(1)

        # Projection to same dimension
        self.proj_eff = nn.Linear(1280, 512)
        self.proj_res = nn.Linear(2048, 512)

        # Cross Attention
        self.attn = nn.MultiheadAttention(
            embed_dim=512,
            num_heads=8,
            batch_first=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        eff_f = self.pool(self.eff(x)).flatten(1)
        res_f = self.pool(self.res(x)).flatten(1)

        eff_f = self.proj_eff(eff_f).unsqueeze(1)
        res_f = self.proj_res(res_f).unsqueeze(1)

        # ResNet queries EfficientNet
        fused, _ = self.attn(res_f, eff_f, eff_f)

        return self.classifier(fused.squeeze(1))


**ResNet18_EffNetB0_Hybrid**

In [39]:
class ResNet18_EffNetB0_Hybrid(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super().__init__()

        self.res = nn.Sequential(*list(models.resnet18(pretrained=True).children())[:-2])
        self.eff = models.efficientnet_b0(pretrained=True).features

        self.pool = nn.AdaptiveAvgPool2d(1)

        fused_dim = 512 + 1280

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        for p in self.res.parameters():
            p.requires_grad = False
        for p in self.eff.parameters():
            p.requires_grad = False

    def forward(self, x):
        r = self.pool(self.res(x)).flatten(1)
        e = self.pool(self.eff(x)).flatten(1)

        fused = torch.cat([r, e], dim=1)
        return self.classifier(fused)


**DenseNet121_MobileNetV2**

In [40]:
class DenseNet121_MobileNetV2(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super().__init__()

        self.den = models.densenet121(pretrained=True).features
        self.mob = models.mobilenet_v2(pretrained=True).features

        self.pool = nn.AdaptiveAvgPool2d(1)

        fused_dim = 1024 + 1280

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        for p in self.den.parameters():
            p.requires_grad = False
        for p in self.mob.parameters():
            p.requires_grad = False

    def forward(self, x):
        d = self.pool(self.den(x)).flatten(1)
        m = self.pool(self.mob(x)).flatten(1)

        fused = torch.cat([d, m], dim=1)
        return self.classifier(fused)


***MY MOdEL***

In [24]:
# ===============================
# ASTRA CONV NET - ÖZGÜN MODEL
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------
# Depthwise Separable Convolution Bloğu
# -------------------------------------------------
class DSConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dilation=1):
        super().__init__()

        self.depthwise = nn.Conv2d(
            in_ch, in_ch,
            kernel_size=3,
            stride=stride,
            padding=dilation,
            dilation=dilation,
            groups=in_ch,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_ch, out_ch,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.Mish()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.act(x)

# -------------------------------------------------
# Channel Attention (Özgün ve sade yapı)
# -------------------------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        y = F.mish(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

# -------------------------------------------------
# Residual Dilated Block (doku yakalama için)
# -------------------------------------------------
class ResidualDilatedBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = DSConv(channels, channels, dilation=1)
        self.conv2 = DSConv(channels, channels, dilation=2)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))

# -------------------------------------------------
# ASTRA CONV NET (ANA MODEL)
# -------------------------------------------------
class AstraConvNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # -------- Stem --------
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.Mish()
        )

        # -------- Multi-Scale Branches --------
        self.branch1 = nn.Sequential(
            DSConv(32, 64),
            ResidualDilatedBlock(64)
        )

        self.branch2 = nn.Sequential(
            nn.AvgPool2d(2),
            DSConv(32, 64),
            ResidualDilatedBlock(64)
        )

        self.branch3 = nn.Sequential(
            nn.AvgPool2d(4),
            DSConv(32, 64),
            ResidualDilatedBlock(64)
        )

        # -------- Feature Fusion --------
        self.fusion = nn.Sequential(
            DSConv(192, 256),
            ChannelAttention(256)
        )

        # -------- Classifier --------
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.Mish(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)

        b1 = self.branch1(x)
        b2 = F.interpolate(self.branch2(x), size=b1.shape[2:], mode="bilinear", align_corners=False)
        b3 = F.interpolate(self.branch3(x), size=b1.shape[2:], mode="bilinear", align_corners=False)

        x = torch.cat([b1, b2, b3], dim=1)
        x = self.fusion(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)

        return self.classifier(x)

# -------------------------------------------------
# Model Test
# -------------------------------------------------
if __name__ == "__main__":
    model = AstraConvNet(num_classes=4)
    x = torch.randn(8, 1, 224, 224)
    y = model(x)
    print("Output shape:", y.shape)


Output shape: torch.Size([8, 4])


In [41]:
def get_transfer_model(model_name):
    model = getattr(models, model_name)(pretrained=True)

    for param in model.parameters():
        param.requires_grad = False

    if "resnet" in model_name:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        model.classifier[-1] = nn.Linear(
            model.classifier[-1].in_features, num_classes
        )

    return model.to(device)


In [42]:
import copy

def train_model(model, optimizer, scheduler, patience=20):
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_loss = float('inf')
    early_stop_counter = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(EPOCHS):
        model.train()
        correct, total, loss_sum = 0, 0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()
            _, pred = torch.max(out, 1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_acc = correct / total

        model.eval()
        correct, total, val_loss = 0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item()
                _, pred = torch.max(out, 1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        train_loss_avg = loss_sum / len(train_loader)
        val_loss_avg = val_loss / len(val_loader)

        if scheduler:
            scheduler.step(val_loss_avg)

        history["train_loss"].append(train_loss_avg)
        history["val_loss"].append(val_loss_avg)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss_avg:.4f} - Val Loss: {val_loss_avg:.4f} - Train Acc: {train_acc:.3f} - Val Acc: {val_acc:.3f}")

        if val_loss_avg < best_loss:
            best_loss = val_loss_avg
            best_model_wts = copy.deepcopy(model.state_dict())
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"  EarlyStopping counter: {early_stop_counter} out of {patience}")
            if early_stop_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs! Loading best model weights.")
                model.load_state_dict(best_model_wts)
                break

    if early_stop_counter < patience:
        model.load_state_dict(best_model_wts)

    return history

In [45]:
import ipywidgets as widgets
from IPython.display import display

models_dict = {
    "EffNetB0_MobileNetV2_Hybrid": EfficientNet_MobileNet_Hybrid(num_classes).to(device),
    "MultiScale_EffNet": MultiScale_EffNet(num_classes).to(device),
    "EffNet_CBAM_LSTM": EffNet_CBAM_LSTM(num_classes).to(device),
    "EffNet_ResNet_CrossAttention": EffNet_ResNet_CrossAttention(num_classes).to(device),
    "DenseNet_SE_LSTM": DenseNet_SE_LSTM(num_classes).to(device),
    "EffNet_SpatialAttention_Residual": EffNet_SpatialAttention_Residual(num_classes).to(device),
    "Hierarchical_CNN_AttnPool": Hierarchical_CNN_AttnPool(num_classes).to(device),
    "ResNet18_EffNetB0_Hybrid": ResNet18_EffNetB0_Hybrid(num_classes).to(device),
    "AstraConvNet": AstraConvNet(num_classes).to(device),
}

print("⬇️ Lütfen eğitmek istediğiniz modeli listeden seçiniz:")
model_dropdown = widgets.Dropdown(
    options=list(models_dict.keys()),
    value=list(models_dict.keys())[0],
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(model_dropdown)

⬇️ Lütfen eğitmek istediğiniz modeli listeden seçiniz:


Dropdown(description='Model:', layout=Layout(width='500px'), options=('EffNetB0_MobileNetV2_Hybrid', 'MultiSca…

In [46]:
import gc
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score, precision_score, recall_score, f1_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize
import pandas as pd
import seaborn as sns
from itertools import cycle
import torch
import numpy as np

def plot_history(hist, title, save_path=None):
    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(hist["train_acc"])
    plt.plot(hist["val_acc"])
    plt.title(f"{title} Accuracy")
    plt.legend(["Train", "Val"])

    plt.subplot(1,2,2)
    plt.plot(hist["train_loss"])
    plt.plot(hist["val_loss"])
    plt.title(f"{title} Loss")
    plt.legend(["Train", "Val"])

    if save_path:
        plt.savefig(save_path)
        print(f"History plot saved to {save_path}")

    plt.show()

def evaluate_model(model, loader, name, history=None):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            prob = torch.softmax(out, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(torch.argmax(out, 1).cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    # Calculate Basic Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)

    try:
        y_true_bin = label_binarize(y_true, classes=range(num_classes))
        auc_val = roc_auc_score(y_true_bin, np.array(y_prob), multi_class='ovr', average='macro')
    except Exception as e:
        print(f"AUC Calculation Error: {e}")
        auc_val = 0

    cm = confusion_matrix(y_true, y_pred)
    specificities = []
    for i in range(num_classes):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(spec)
    avg_spec = np.mean(specificities)

    print(f"Test Accuracy: {acc:.4f}")

    base_save_dir = "/content/drive/MyDrive/brain_tumor/models"
    folder_name = os.path.join(base_save_dir, f"{name}_{acc:.4f}")
    os.makedirs(folder_name, exist_ok=True)
    print(f"Results will be saved in: {folder_name}")

    metrics_data = {
        "Metric": ["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1 Score (Macro)", "Cohen's Kappa", "AUC (Macro)", "Specificity (Macro)"],
        "Value": [f"{acc:.4f}", f"{prec:.4f}", f"{rec:.4f}", f"{f1:.4f}", f"{kappa:.4f}", f"{auc_val:.4f}", f"{avg_spec:.4f}"]
    }
    df_metrics = pd.DataFrame(metrics_data)

    plt.figure(figsize=(8, 4))
    plt.axis('tight')
    plt.axis('off')
    table = plt.table(cellText=df_metrics.values, colLabels=df_metrics.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title(f"{name} Detailed Metrics")
    plt.savefig(os.path.join(folder_name, 'metrics_table.png'), bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{name} Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(folder_name, 'confusion_matrix.png'))
    plt.show()

    report = classification_report(y_true, y_pred, target_names=class_names)
    print(report)
    with open(os.path.join(folder_name, 'classification_report.txt'), 'w') as f:
        f.write(report)

    if history:
        plot_history(history, name, save_path=os.path.join(folder_name, 'history.png'))


    y_prob = np.array(y_prob)
    fpr, tpr, roc_auc = {}, {}, {}
    precision, recall, pr_auc = {}, {}, {}

    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        precision[i], recall[i], _ = precision_recall_curve(y_true_bin[:, i], y_prob[:, i])
        pr_auc[i] = average_precision_score(y_true_bin[:, i], y_prob[:, i])

    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple'])
    for i, color in zip(range(num_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.title(f'{name} ROC Curve')
    plt.legend(loc="lower right")

    plt.subplot(1, 2, 2)
    for i, color in zip(range(num_classes), colors):
        plt.plot(recall[i], precision[i], color=color, lw=2,
                 label=f'{class_names[i]} (AP = {pr_auc[i]:.2f})')
    plt.title(f'{name} Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(os.path.join(folder_name, 'roc_pr_curves.png'))
    plt.show()

    # Save JIT
    try:
        print(f"Saving {name} with TorchScript (JIT)...")
        dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        traced_model = torch.jit.trace(model, dummy_input)
        jit_path = os.path.join(folder_name, f"{name}_jit.pt")
        traced_model.save(jit_path)
        print(f"Model saved successfully to {jit_path}")
    except Exception as e:
        print(f"Error saving with JIT: {e}")

if 'histories' not in globals():
    histories = {}

selected_name = model_dropdown.value
print(f"Model: {selected_name}")


model = models_dict[selected_name]

print(f"\n Training {selected_name}")


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

histories[selected_name] = train_model(model, optimizer, scheduler)
plot_history(histories[selected_name], selected_name)
print(f" {selected_name} on Test Set...")
evaluate_model(model, test_loader, selected_name, history=histories.get(selected_name))

Model: AstraConvNet

 Training AstraConvNet


RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[64, 3, 224, 224] to have 1 channels, but got 3 channels instead